In [1]:
import pandas as pd
import feature_engineering_helper as hf
import importlib
from sklearn.model_selection import StratifiedShuffleSplit

In [2]:
# get raw data
data_prefix = '../0_data/processed_data/'
original_train = pd.read_csv(data_prefix + 'train_without_data_augmentation.csv')[['SMILES','MP']]

non_feature_cols = ['SMILES', 'MP', 'Type', 'MP_label']

original_test = pd.read_csv(data_prefix + 'test_predictions.csv')[['SMILES','exp MP']]
original_test = original_test.rename(columns={'exp MP': 'MP'})

raw_data = pd.concat([original_train, original_test], axis=0).reset_index(drop=True)

print(original_train.shape)
print(original_test.shape)
print(raw_data.shape)

(17633, 2)
(1961, 2)
(19594, 2)


In [3]:
# Clean data with new melting point bounds (between low_bound and high_bound)
clean_data = hf.clean_dataset(raw_data, low_bound=0, high_bound=500)

Initial shape: (19594, 2)
Removed 2370 rows with MP < 0
Removed 2 rows with MP > 500
Final shape: (17222, 2)


In [4]:
# add H/L labels based on melting point threshold = 250
clean_data['MP_label'] = clean_data['MP'].apply(lambda x: 'H' if x >= 250 else 'L')

# how many molecules in percentage are labeled as H and L
h_percentage = 100 * (clean_data['MP_label'] == 'H').mean()
l_percentage = 100 * (clean_data['MP_label'] == 'L').mean()
print(f"{h_percentage:.2f}% of molecules are labeled as H")
print(f"{l_percentage:.2f}% of molecules are labeled as L")
clean_data.describe()

5.10% of molecules are labeled as H
94.90% of molecules are labeled as L


,MP
count,17222.000000
mean,126.045995
std,70.913439
min,0.000000
25%,69.000000
50%,120.000000
75%,175.000000
max,492.500000


In [5]:
data_with_features = hf.smiles_to_features(clean_data, ['rdkit', 'maccs'])
data_with_features

/Users/zeqing/0_Github/melting_point_2026/1_feature_engineering/feature_engineering_helper.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_with_features[f'RDKit_{name}'] = [f[i] for f in rdkit_features]


✓ RDKit: Added 217 features
✓ MACCS: Added 167 features


/Users/zeqing/0_Github/melting_point_2026/1_feature_engineering/feature_engineering_helper.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_with_features[f'MACCS_{i}'] = maccs_array[:, i]


,MACCS_0,MACCS_1,MACCS_10,MACCS_100,MACCS_101,MACCS_102,MACCS_103,MACCS_104,MACCS_105,MACCS_106,...,RDKit_fr_sulfone,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,SMILES
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.461644,CCOC(=O)/C=C/C(=O)OCC
1,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0.322574,O=C(c1ccc(cc1)C(=O)Oc1cc(Cl)cc(c1)Cl)Oc1cc(Cl)...
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.344216,O=C(c1ccccc1)Nc1cccc(c1)/N=N/c1cccc(c1)NC(=O)c...
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.609080,OCc1cccc(n1)CO
5,0,0,0,0,1,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0.779142,COc1ccc(cc1)c1nnc2n1NC(S2)c1ccc(cc1)Cl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19588,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0.735003,OC(=O)Cc1ccc(c(c1)F)F
19589,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0.812970,CCOC(=O)N(c1ccccc1)c1ccccc1
19591,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0.805020,OC(=O)Cc1ccc(c(c1)Cl)Cl
19592,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.519957,CCCOC(=O)c1cc(O)c(c(c1)O)O


In [6]:
# drop rows with NaN values and print how many rows were dropped and reset index
nan_rows = data_with_features[data_with_features.isna().any(axis=1)]
num_dropped = nan_rows.shape[0]
print(f"Dropped {num_dropped} rows containing NaN values.")
data_with_features = data_with_features.dropna().reset_index(drop=True)

Dropped 2 rows containing NaN values.


In [7]:
label = 'MP_label'

# stratified split based on label compliance

split = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
for train_index, test_index in split.split(data_with_features, data_with_features[label]):
    strat_train_set = data_with_features.loc[train_index]
    strat_test_set = data_with_features.loc[test_index]
    strat_train_set['Type'] = 'Train'
    strat_test_set['Type'] = 'Test'
final_data = pd.concat([strat_train_set, strat_test_set], axis=0).reset_index(drop=True)

# print the shape of train and test dataset
print(f"Train set: {final_data[final_data['Type'] == 'Train'].shape}")
print(f"Test set: {final_data[final_data['Type'] == 'Test'].shape}")

# print the percentage of label violations in train and test sets
train_violation_percentage = 100 * (final_data[final_data['Type'] == 'Train'][label] == 'H').mean()
test_violation_percentage = 100 * (final_data[final_data['Type'] == 'Test'][label] == 'H').mean()
print(f"Train set: {train_violation_percentage:.2f}% of molecules are labeled as H")
print(f"Test set: {test_violation_percentage:.2f}% of molecules are labeled as H")


Train set: (12054, 388)
Test set: (5166, 388)
Train set: 5.10% of molecules are labeled as H
Test set: 5.09% of molecules are labeled as H


/var/folders/hk/7jjnrbrj53n1t8_bmkhf0_k00000gn/T/ipykernel_27788/1921117180.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  strat_train_set['Type'] = 'Train'
/var/folders/hk/7jjnrbrj53n1t8_bmkhf0_k00000gn/T/ipykernel_27788/1921117180.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  strat_test_set['Type'] = 'Test'


In [8]:
final_data.to_parquet(data_prefix + 'data_with_all_features.parquet', compression= "zstd")

In [9]:
data_with_all_features_scaled = hf.standardize_data(non_feature_cols=non_feature_cols, dataframe=final_data, scaler_path=data_prefix + 'data_with_all_features_scaled_scaler.pkl')
data_with_all_features_scaled.to_parquet(data_prefix + 'data_with_all_features_scaled.parquet', compression= "zstd")



Number of feature columns to standardize: 384
Scaler saved to ../0_data/processed_data/data_with_all_features_scaled_scaler.pkl


In [10]:
data_with_all_features_scaled[data_with_all_features_scaled['Type'] == 'Train'].describe()

,MACCS_0,MACCS_1,MACCS_10,MACCS_100,MACCS_101,MACCS_102,MACCS_103,MACCS_104,MACCS_105,MACCS_106,...,RDKit_fr_sulfonamd,RDKit_fr_sulfone,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed
count,12054.0,12054.0,12054.0,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,...,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04,1.205400e+04
mean,0.0,0.0,0.0,4.597838e-17,2.593652e-17,-3.183118e-17,-8.841995e-18,5.069411e-17,6.587286e-17,-5.776770e-17,...,6.189397e-17,3.934688e-17,-3.301012e-17,-1.650506e-17,-2.298919e-17,3.536798e-18,5.894663e-18,-1.886292e-17,-4.244158e-17,-9.195675e-17
std,0.0,0.0,0.0,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,...,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00
min,0.0,0.0,0.0,-3.726780e-01,-6.229527e-01,-4.760468e-01,-4.378304e-01,-4.336524e-01,-6.427520e-01,-4.849408e-01,...,-1.357348e-01,-1.024151e-01,-7.260094e-02,-5.842062e-02,-1.282574e-01,-1.721550e-02,-1.218433e-01,-1.682288e-01,-1.476669e-01,-3.469270e+00
25%,0.0,0.0,0.0,-3.726780e-01,-6.229527e-01,-4.760468e-01,-4.378304e-01,-4.336524e-01,-6.427520e-01,-4.849408e-01,...,-1.357348e-01,-1.024151e-01,-7.260094e-02,-5.842062e-02,-1.282574e-01,-1.721550e-02,-1.218433e-01,-1.682288e-01,-1.476669e-01,-6.110821e-01
50%,0.0,0.0,0.0,-3.726780e-01,-6.229527e-01,-4.760468e-01,-4.378304e-01,-4.336524e-01,-6.427520e-01,-4.849408e-01,...,-1.357348e-01,-1.024151e-01,-7.260094e-02,-5.842062e-02,-1.282574e-01,-1.721550e-02,-1.218433e-01,-1.682288e-01,-1.476669e-01,8.082101e-02
75%,0.0,0.0,0.0,-3.726780e-01,1.605258e+00,-4.760468e-01,-4.378304e-01,-4.336524e-01,1.555810e+00,-4.849408e-01,...,-1.357348e-01,-1.024151e-01,-7.260094e-02,-5.842062e-02,-1.282574e-01,-1.721550e-02,-1.218433e-01,-1.682288e-01,-1.476669e-01,7.242889e-01
max,0.0,0.0,0.0,2.683282e+00,1.605258e+00,2.100634e+00,2.283989e+00,2.305994e+00,1.555810e+00,2.062108e+00,...,1.245001e+01,1.903730e+01,4.660109e+01,1.711724e+01,1.540958e+01,8.298906e+01,2.116365e+01,1.830345e+01,1.328612e+01,2.181077e+00
